In [1]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm

In [4]:
BASE_FILE = os.path.join("..", "..", "data", "processed", "cardio_onc_prostate_06_broad_clean.csv")

df = pd.read_csv(BASE_FILE)

print(df.shape)
df.head()

(239, 60)


,unique_patient_id,ethnicity,nht_auth_date,nht_start_date,bmi,specific_nht_used,age,adt_start_date,adt_agent,hx_smoking,...,bp_meds_post_binary,lipid_meds_post_binary,dm_meds_post_binary,at_risk,lifestyle_counseling,dm_severity,ethnicity_enc,specific_nht_used_enc,adt_agent_enc,prescribing_provider_enc
0,1.0,NaN,2022-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,27.0
1,2.0,NaN,2022-01-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,27.0
2,3.0,Caucasian,2022-01-12,2022-02-01,25.99,Darolutamide,73.0,2018-01-26,Lupron,1.0,...,0.0,0.0,0.0,0.0,0,0.0,2.0,2.0,2.0,27.0
3,4.0,Asian,2022-01-14,2022-02-14,22.55,Apalutamide,93.0,2021-12-01,Bicalutamide,0.0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,1.0,0.0,42.0
4,5.0,NaN,2022-01-19,NaN,NaN,Abiraterone,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0.0,NaN,7.0


## NHT analysis

In [18]:
# Which features are related to NHT choice?

continuous_features = [
    "bmi", "age", "sbp", "dbp", "ascvd_10yr",
    "days_auth_to_start", "days_adt_to_nht",
]

binary_features = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done", "non_onc_provider",
]

encoded_features = [
    "ethnicity_enc",
    "adt_agent_enc", "prescribing_provider_enc",
]

all_features = continuous_features + binary_features + encoded_features


results = []

for var in all_features:
    try:
        df_tmp = df[["specific_nht_used", var]].dropna()

        X = sm.add_constant(df_tmp[[var]])
        y = df_tmp["specific_nht_used"].astype(str)

        model = sm.MNLogit(y, X).fit(disp=0)

        results.append({
            "feature": var,
            "p_value": model.llr_pvalue
        })

    except:
        continue

df_result = pd.DataFrame(results)

df_result[df_result["p_value"] < 1].sort_values("p_value")

c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximu

,feature,p_value
1,age,0.000388
38,ethnicity_enc,0.026749
14,statin_prior,0.030953
39,adt_agent_enc,0.038651
26,on_insulin,0.043954
4,ascvd_10yr,0.044037
6,family_hx_hd,0.071565
20,hx_arrhythmia,0.099938
11,has_pcp,0.119730
24,hx_dm2,0.125231


In [9]:
median_ages = (
    df.groupby("specific_nht_used")["age"]
    .median()
    .sort_values(ascending=False)
)

print("\nMedian age by NHT agent:")
print(median_ages.round(1))


Median age by NHT agent:
specific_nht_used
Darolutamide    75.0
Apalutamide     73.5
Enzalutamide    71.0
Abiraterone     69.0
Name: age, dtype: float64


In [11]:
# On insulin -> NHT received

df_tmp = df[["specific_nht_used", "on_insulin"]].dropna()

X = sm.add_constant(df_tmp[["on_insulin"]])
y = df_tmp["specific_nht_used"].astype(str)

model = sm.MNLogit(y, X).fit(disp=0)

print(model.summary())

                          MNLogit Regression Results                          
Dep. Variable:      specific_nht_used   No. Observations:                  200
Model:                        MNLogit   Df Residuals:                      194
Method:                           MLE   Df Model:                            3
Date:                Fri, 08 May 2026   Pseudo R-squ.:                 0.01794
Time:                        00:22:00   Log-Likelihood:                -221.76
converged:                       True   LL-Null:                       -225.81
Covariance Type:            nonrobust   LLR p-value:                   0.04395
 specific_nht_used=Apalutamide       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                             -1.9095      0.309     -6.173      0.000      -2.516      -1.303
on_insulin                        -0.0364      1.113     -0.033      0.

In [16]:
# Statin Prior -> NHT received

df_tmp = df[["specific_nht_used", "statin_prior"]].dropna()

X = sm.add_constant(df_tmp[["statin_prior"]])
y = df_tmp["specific_nht_used"].astype(str)

model = sm.MNLogit(y, X).fit(disp=0)

print(model.summary())

                          MNLogit Regression Results                          
Dep. Variable:      specific_nht_used   No. Observations:                  203
Model:                        MNLogit   Df Residuals:                      197
Method:                           MLE   Df Model:                            3
Date:                Fri, 08 May 2026   Pseudo R-squ.:                 0.01943
Time:                        00:31:44   Log-Likelihood:                -224.01
converged:                       True   LL-Null:                       -228.45
Covariance Type:            nonrobust   LLR p-value:                   0.03095
 specific_nht_used=Apalutamide       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                             -1.9661      0.404     -4.872      0.000      -2.757      -1.175
statin_prior                       0.0943      0.596      0.158      0.

In [19]:
# Arrhythmia -> NHT received

df_tmp = df[["specific_nht_used", "hx_arrhythmia"]].dropna()

X = sm.add_constant(df_tmp[["hx_arrhythmia"]])
y = df_tmp["specific_nht_used"].astype(str)

model = sm.MNLogit(y, X).fit(disp=0)

print(model.summary())

                          MNLogit Regression Results                          
Dep. Variable:      specific_nht_used   No. Observations:                  203
Model:                        MNLogit   Df Residuals:                      197
Method:                           MLE   Df Model:                            3
Date:                Fri, 08 May 2026   Pseudo R-squ.:                 0.01369
Time:                        00:32:39   Log-Likelihood:                -225.33
converged:                       True   LL-Null:                       -228.45
Covariance Type:            nonrobust   LLR p-value:                   0.09994
 specific_nht_used=Apalutamide       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                             -1.9716      0.322     -6.126      0.000      -2.602      -1.341
hx_arrhythmia                      0.3621      0.839      0.432      0.

## ADT analysis

In [20]:
# Which features are related to ADT choice?

continuous_features = [
    "bmi", "age", "sbp", "dbp", "ascvd_10yr",
    "days_auth_to_start", "days_adt_to_nht",
]

binary_features = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done", "non_onc_provider",
]

encoded_features = [
    "ethnicity_enc",
    "adt_agent_enc", "prescribing_provider_enc",
]

all_features = continuous_features + binary_features + encoded_features


results = []

for var in all_features:
    try:
        df_tmp = df[["adt_agent", var]].dropna()

        X = sm.add_constant(df_tmp[[var]])
        y = df_tmp["adt_agent"].astype(str)

        model = sm.MNLogit(y, X).fit(disp=0)

        results.append({
            "feature": var,
            "p_value": model.llr_pvalue
        })

    except:
        continue

df_result = pd.DataFrame(results)

df_result[df_result["p_value"] < 0.05].sort_values("p_value")

c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Trian\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximu

,feature,p_value
16,lipid_panel_checked,0.003732
31,cards_post,0.004770
13,hx_high_tg,0.010041
8,bp_meds_prior,0.013833
17,hx_cad,0.016638
34,exercise_counseling,0.027958
2,sbp,0.031748
35,echo_ordered,0.033174


In [25]:
# bp_meds_prior -> ADT

df_tmp = df[["adt_agent", "bp_meds_prior"]].dropna()

X = sm.add_constant(df_tmp[["bp_meds_prior"]])
y = df_tmp["adt_agent"].astype(str)

model = sm.MNLogit(y, X).fit(disp=0)

print(model.summary())

                          MNLogit Regression Results                          
Dep. Variable:              adt_agent   No. Observations:                  192
Model:                        MNLogit   Df Residuals:                      186
Method:                           MLE   Df Model:                            3
Date:                Fri, 08 May 2026   Pseudo R-squ.:                 0.02259
Time:                        00:41:05   Log-Likelihood:                -230.17
converged:                       True   LL-Null:                       -235.49
Covariance Type:            nonrobust   LLR p-value:                   0.01383
adt_agent=Firmagon       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.4098      0.446      0.918      0.358      -0.465       1.284
bp_meds_prior         -0.9574      0.402     -2.380      0.017      -1.746      -0.169
--------------------

In [26]:
# hx_cad -> ADT

df_tmp = df[["adt_agent", "hx_cad"]].dropna()

X = sm.add_constant(df_tmp[["hx_cad"]])
y = df_tmp["adt_agent"].astype(str)

model = sm.MNLogit(y, X).fit(disp=0)

print(model.summary())

                          MNLogit Regression Results                          
Dep. Variable:              adt_agent   No. Observations:                  191
Model:                        MNLogit   Df Residuals:                      185
Method:                           MLE   Df Model:                            3
Date:                Fri, 08 May 2026   Pseudo R-squ.:                 0.02182
Time:                        00:42:42   Log-Likelihood:                -229.53
converged:                       True   LL-Null:                       -234.65
Covariance Type:            nonrobust   LLR p-value:                   0.01664
adt_agent=Firmagon       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.3054      0.352     -0.867      0.386      -0.996       0.385
hx_cad                -0.3878      0.706     -0.549      0.583      -1.772       0.997
--------------------

In [28]:
# sbp -> ADT

df_tmp = df[["adt_agent", "sbp"]].dropna()

X = sm.add_constant(df_tmp[["sbp"]])
y = df_tmp["adt_agent"].astype(str)

model = sm.MNLogit(y, X).fit(disp=0)

print(model.summary())

                          MNLogit Regression Results                          
Dep. Variable:              adt_agent   No. Observations:                  141
Model:                        MNLogit   Df Residuals:                      135
Method:                           MLE   Df Model:                            3
Date:                Fri, 08 May 2026   Pseudo R-squ.:                 0.02748
Time:                        00:45:02   Log-Likelihood:                -156.12
converged:                       True   LL-Null:                       -160.53
Covariance Type:            nonrobust   LLR p-value:                   0.03175
adt_agent=Firmagon       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  4.3912      3.414      1.286      0.198      -2.300      11.083
sbp                   -0.0372      0.026     -1.437      0.151      -0.088       0.014
--------------------